# 🪰 Fly Brain Starter

You're getting a real fruit fly's brain: all **139,000 neurons**, wired exactly like the real thing. Then you'll simulate it.

**How to use this:**
1. **File → Save a copy in Drive** (so you can edit)
2. Run each cell top to bottom (Shift + Enter)
3. Build something with it

Takes about 5 minutes. Stuck? Grab an organizer.

## 1. Install

In [1]:
!pip install -q fafbseg==3.2.3 navis==1.12.0 plotly crackle-codec

## 2. Your FlyWire token
Get one here (sign in with Google): **https://global.daf-apis.com/auth/api/v1/create_token**

Paste it when asked. It won't be saved in the notebook, so you can share this safely.

In [ ]:
from getpass import getpass
from fafbseg import flywire

flywire.set_chunkedgraph_secret(getpass(""), overwrite=True)
flywire.set_default_dataset("public")

## 3. Look at a neuron

(This may take up to 3 minutes to load in)

This is the **Giant Fibre**, the neuron that makes a fly jump away from danger. Drag to spin it.

In [ ]:
import navis

giant_fibre = flywire.search_annotations(flywire.NeuronCriteria(type="DNp01"))
print(giant_fibre[["root_id", "cell_type", "side"]])
navis.plot3d(flywire.get_skeletons(giant_fibre.root_id.iloc[0]))

Using annotation version "latest tagged release" (a83b277 from 2026-09-29) from https://github.com/flyconnectome/flywire_annotations.
Caching flywire_annotations@a83b2776d60d5764cef36b927f5f9679c16c47a2.tsv from github.com... Done.
Found 2 neurons matching the given criteria.
Using materialization version 783.
Caching root IDs for hierarchical annotations at materialization '783'... Done.
              root_id cell_type   side
0  720575940632499757     DNp01  right
1  720575940622838154     DNp01   left


INFO  : Using "plotly" backend for 3D plotting. (navis)
INFO:navis:Using "plotly" backend for 3D plotting.


## 4. Load the whole brain
Three files, no token needed:
- **neurons**: every neuron's ID
- **wiring**: every connection, as +synapses (excites) or −synapses (inhibits)
- **labels**: what each neuron is (sugar taste, escape, ...)

In [ ]:
import pandas as pd, numpy as np, scipy.sparse as sp

SHIU = "https://github.com/philshiu/Drosophila_brain_model/raw/main/"
neurons = pd.read_csv(SHIU + "Completeness_783.csv", index_col=0)
wiring = pd.read_parquet(SHIU + "Connectivity_783.parquet")
labels = pd.read_csv("https://github.com/flyconnectome/flywire_annotations/raw/main/supplemental_files/"
                     "Supplemental_file1_neuron_annotations.tsv", sep="\t", low_memory=False)

N = len(neurons)
W = sp.csr_matrix((wiring["Excitatory x Connectivity"],
                   (wiring["Presynaptic_Index"], wiring["Postsynaptic_Index"])), shape=(N, N))
row = {root_id: i for i, root_id in enumerate(neurons.index)}

def find(mask):
    """Neurons matching a label filter -> their row numbers in W."""
    return np.array([row[r] for r in labels.root_id[mask] if r in row])

print(f"{N:,} neurons, {W.nnz:,} connections")

138,639 neurons, 15,091,983 connections


## 5. Pick the neurons we care about
| Group | What it does |
|---|---|
| sugar | taste cells on the mouth that detect sugar |
| bitter | taste cells that detect bitter |
| looming | eye cells that see something rushing at you |
| mn9 | motor neuron that sticks the mouth out to **eat** |
| giant_fibre | the **escape** jump command |
| mdn | makes the fly **walk backwards** |

Want others? Browse names at **https://codex.flywire.ai** and filter `labels` the same way.

In [ ]:
sugar       = find(labels.cell_type == "LB3c")
bitter      = find(labels.cell_sub_class == "bitter")
looming     = find(labels.cell_type.isin(["LC4", "LPLC2"]))
giant_fibre = find(labels.cell_type == "DNp01")
mdn         = find(labels.cell_type == "MDN")
mn9         = np.array([row[720575940660219265]])  # not in the label file; ID from Shiu et al. 2024

for name in ["sugar", "bitter", "looming", "mn9", "giant_fibre", "mdn"]:
    print(f"{name:12s} {len(globals()[name]):4d} neurons")

sugar          32 neurons
bitter         42 neurons
looming       314 neurons
mn9             1 neurons
giant_fibre     2 neurons
mdn             4 neurons


## 6. The simulator
Every neuron is a tiny voltage meter. Incoming spikes push it up (or down, if inhibitory), it leaks back toward rest, and when it crosses a threshold it **spikes** and passes that on. Settings come from [Shiu et al. 2024](https://www.nature.com/articles/s41586-024-07763-9).

You don't need to read this code. Just run it.

In [ ]:
def simulate(inputs, ms=200, W=W, seed=0):
    """inputs: [(neurons, rate_hz), ...]  ->  spike count for every neuron."""
    dt, v0, vth, tm, ts, ref = 0.1, -52.0, -45.0, 20.0, 5.0, 2.2
    wsyn, kick, delay = 0.275, 250 * 0.275, int(1.8 / 0.1)
    rng = np.random.default_rng(seed)
    v, g = np.full(N, v0), np.zeros(N)
    last, counts = np.full(N, -1e9), np.zeros(N, int)
    driven = np.zeros(N, bool)
    for idx, _ in inputs:
        driven[idx] = True
    ring = np.zeros((delay, N))
    for k in range(int(ms / dt)):
        t, slot = k * dt, k % delay
        g += ring[slot]; ring[slot] = 0
        free = ((t - last) > ref) | driven
        v[free] += dt * (v0 - v[free] + g[free]) / tm
        g[free] *= np.exp(-dt / ts)
        for idx, hz in inputs:
            v[idx[rng.random(len(idx)) < hz * dt / 1000]] += kick
        s = np.flatnonzero(v > vth)
        v[s], g[s], last[s] = v0, 0.0, t
        counts[s] += 1
        if len(s):
            ring[slot] += np.asarray(W[s].sum(0)).ravel() * wsyn
    return counts

def rate(counts, group, ms=200):
    """Average spikes per second of a group."""
    return counts[group].sum() / len(group) / (ms / 1000)

## 7. Poke the brain 🧪
Feed it sugar, bitter and a scary looming shadow, and watch what it decides. Each run is 0.2 s of brain time and takes a few seconds.

**What you should see:**
- nothing in → nothing out
- sugar → it wants to **eat** (mn9 fires)
- sugar + bitter → it changes its mind
- looming → it **escapes** (giant_fibre fires)

In [ ]:
experiments = {
    "nothing":        [],
    "sugar":          [(sugar, 200)],
    "sugar + bitter": [(sugar, 200), (bitter, 200)],
    "looming":        [(looming, 150)],
}
results = []
for name, inputs in experiments.items():
    c = simulate(inputs)
    results.append({"experiment": name, "eat (mn9 Hz)": rate(c, mn9),
                    "escape (giant_fibre Hz)": rate(c, giant_fibre),
                    "back up (mdn Hz)": rate(c, mdn), "neurons active": int((c > 0).sum())})
results = pd.DataFrame(results)
results

,experiment,eat (mn9 Hz),escape (giant_fibre Hz),back up (mdn Hz),neurons active
0,nothing,0.0,0.0,0.00,0
1,sugar,100.0,0.0,0.00,419
2,sugar + bitter,5.0,0.0,0.00,296
3,looming,0.0,265.0,16.25,1296


## What now?
Swap the fly's world for yours. Your signal goes into a sense, and a behaviour comes out:
- **Game bot:** food → `sugar`, enemy → `looming`. `mn9` = go for it, `giant_fibre` = dodge
- **Trading bot:** price up → `sugar`, crash → `looming`. `giant_fibre` = sell everything
- **Lesion lab:** cut a neuron type's outputs (it still fires, but nothing hears it) and see what breaks: `keep = np.ones(N); keep[giant_fibre] = 0; simulate(inputs, W=(sp.diags(keep) @ W).tocsr())`
- **Give it a body:** [flybody](https://github.com/TuragaLab/flybody) is a physics-simulated fly for animations

The handbook has more ideas and every resource link.